# Earthquake detection — YOLOv8s (scaffold)

Self-contained Colab notebook (same recipe as the fire / car-crash notebooks) for training a **earthquake detector**. It is a **scaffold**: you supply a YOLO-format dataset from Roboflow Universe — the author could not pin a specific public dataset, so the dataset slug below is a placeholder you fill in.

1. Pull a YOLOv8 dataset from Roboflow Universe (search: "earthquake building damage", "post-earthquake damage detection", "collapsed building", "structural damage")
2. Train **YOLOv8s** for 30 epochs on a free T4, saving to Google Drive
3. Evaluate on the test split
4. Sanity-check on your own uploaded images
5. Package `earthquake.pt` for download

**Runtime:** free T4 GPU (Runtime → Change runtime type → T4 GPU).
**You need a free Roboflow API key:** https://app.roboflow.com → avatar → Settings → Roboflow API → Private API Key.

## 0. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.3.40 roboflow Pillow

## Mount Google Drive (persists training runs)

Saves every checkpoint straight to Drive so a Colab disconnect doesn't lose the model. If the runtime dies mid-training, re-mount and rerun the train cell with `resume=True`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RUNS = '/content/drive/MyDrive/earthquake_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('Training runs will be saved to:', DRIVE_RUNS)

## 1. Download the dataset from Roboflow Universe

Find a suitable YOLOv8 dataset on https://universe.roboflow.com — search for: **"earthquake building damage", "post-earthquake damage detection", "collapsed building", "structural damage"**.
Open the dataset → *Download this Dataset* → format **YOLOv8** → it shows the `workspace`, `project` and `version`. Paste them below. (No dataset was hard-coded here because availability couldn't be verified — pick one that fits and check its class labels.)

In [ ]:
from getpass import getpass
from pathlib import Path
from roboflow import Roboflow

# ── Fill these in from the dataset's "Download" page on Roboflow Universe ──
WORKSPACE = "<workspace-slug>"
PROJECT   = "<project-slug>"
VERSION   = 1
# ──────────────────────────────────────────────────────────────────────────
assert "<" not in WORKSPACE + PROJECT, "Set WORKSPACE / PROJECT / VERSION above first."

API_KEY = getpass('Roboflow API key: ').strip()
rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8')

DST = Path(dataset.location)
print('Dataset at:', DST)
print('data.yaml :\n', (DST / 'data.yaml').read_text())

### Tidy data.yaml — absolute paths

In [ ]:
import yaml
data_path = DST / 'data.yaml'
data = yaml.safe_load(data_path.read_text())
data['path'] = str(DST)
for split in ('train', 'val', 'test'):
    if split in data and isinstance(data[split], str):
        data[split] = data[split].replace('../', '')
data_path.write_text(yaml.safe_dump(data))
print(data_path.read_text())

### Sanity check: a few labelled samples

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import random

NAMES = data['names'] if isinstance(data['names'], list) else list(data['names'].values())
print('Classes:', NAMES)

img_dir = DST / 'train' / 'images'
lbl_dir = DST / 'train' / 'labels'
pool = list(img_dir.glob('*'))[:200]
samples = random.sample(pool, k=min(6, len(pool)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, samples):
    img = Image.open(img_path).convert('RGB'); W, H = img.size
    draw = ImageDraw.Draw(img)
    lbl_path = lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            c, x, y, w, h = int(parts[0]), *map(float, parts[1:5])
            x1, y1 = (x - w/2) * W, (y - h/2) * H
            x2, y2 = (x + w/2) * W, (y + h/2) * H
            draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
            draw.text((x1, max(0, y1 - 12)), NAMES[c], fill='red')
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Train YOLOv8s (30 epochs, saved to Drive)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data=str(DST / 'data.yaml'),
    epochs=30,
    imgsz=640,
    batch=32,
    patience=15,
    close_mosaic=10,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    project=DRIVE_RUNS,
    name='earthquake_yolov8s',
    device=0,
    workers=2,
    seed=42,
    verbose=True,
)

## 3. Evaluate on the test split

In [ ]:
from pathlib import Path
best = Path(DRIVE_RUNS) / 'earthquake_yolov8s' / 'weights' / 'best.pt'
model = YOLO(str(best))
metrics = model.val(data=str(DST / 'data.yaml'), split='test', imgsz=640, plots=True)
print('mAP@0.5      :', round(float(metrics.box.map50), 4))
print('mAP@0.5:0.95 :', round(float(metrics.box.map), 4))
print('Precision    :', round(float(metrics.box.mp), 4))
print('Recall       :', round(float(metrics.box.mr), 4))

## 4. Sanity-check on your own images

Upload a few earthquake photos when prompted.

In [ ]:
from google.colab import files
from pathlib import Path
TEST_DIR = Path('/content/test_eq'); TEST_DIR.mkdir(exist_ok=True)
print('Upload a few test images:')
up = files.upload()
for fn, blob in up.items():
    (TEST_DIR / fn).write_bytes(blob)

results = model.predict(source=str(TEST_DIR), imgsz=640, conf=0.25, save=True)
for r in results:
    print(r.path, '→', len(r.boxes), 'detections')
    for b in r.boxes:
        print(f'  {model.names[int(b.cls)]} conf={float(b.conf):.2f}')

import glob
from IPython.display import Image as IPyImage, display
for p in sorted(glob.glob(str(Path(results[0].save_dir) / '*'))):
    display(IPyImage(p))

## 5. Package for local use

In [ ]:
import shutil
from google.colab import files

src = Path(DRIVE_RUNS) / 'earthquake_yolov8s'
out = Path('/content/earthquake_model_release'); out.mkdir(exist_ok=True)
shutil.copy(src / 'weights/best.pt', out / 'earthquake.pt')
for f in ['results.png', 'results.csv', 'confusion_matrix.png', 'PR_curve.png']:
    if (src / f).exists():
        shutil.copy(src / f, out / f)

shutil.make_archive('/content/earthquake_model_release', 'zip', out)
files.download('/content/earthquake_model_release.zip')

## Drop the trained weights into the app

Copy `earthquake.pt` to `model/earthquake.pt`. NOTE: unlike fire / flood / car-crash, the app does **not yet** have a earthquake specialist wired in — currently Earthquake is handled at scene-classifier level only. To use this detector you would add a small service + registry entry, using `app/services/fire.py` and `app/services/carcrash.py` as templates (severity + resource heuristics), then register it in `app/api/deps.py` and consult it in `app/api/routes/analyze.py`. This notebook just produces the weights so they're ready when/if that wiring is added.